# Subset-Tagged BGE Reranker Experiment

Kaggle-ready reranker experiment for T4 GPUs.

This notebook tests one controlled change over the current exp2 reranker:

```text
query -> includes subset tag
candidate text -> includes subset tag + candidate question + candidate answer
```

It keeps the same general setup:

- candidate generator: fine-tuned BGE-M3 encoder adapter
- candidate pool: per-subset top50 from Train
- reranker base: `BAAI/bge-reranker-v2-m3`
- training target: `0.75 * ROUGE-1 + 0.25 * ROUGE-L`
- evaluation: Val top1, rerank, oracle@50

Expected runtime on Kaggle T4/T4x2: several hours including installs/model downloads.

In [ ]:
# Optional installs. Leave internet enabled in Kaggle for this cell.
!pip -q install -U "sentence-transformers>=5.1.0" "transformers>=4.46.0" "datasets>=3.0.0" "accelerate>=0.33.0" "peft>=0.12.0" "rouge-score>=0.1.2" "safetensors>=0.4.3"

In [ ]:
from pathlib import Path
import gc
import json
import os
import random
import time

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import PeftModel
from rouge_score import rouge_scorer
from sentence_transformers import CrossEncoder, SentenceTransformer
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.losses import MSELoss
from sklearn.neighbors import NearestNeighbors
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu count", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))

In [ ]:
# Config tuned for Kaggle T4. If OOM, set batch_size=4 and grad_accum=8.
CONFIG = {
    "k": 50,
    "train_pairs_per_query": 12,
    "epochs": 1,
    "batch_size": 4,          # safer for T4; T4x2 may still DataParallel
    "grad_accum": 8,          # effective batch 32
    "lr": 1e-5,
    "max_length": 512,
    "output_dir": Path("/kaggle/working/subset_tag_reranker"),
}
CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
print(CONFIG)

In [ ]:
def find_file(name):
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(root.rglob(name))
    if not matches:
        raise FileNotFoundError(f"Could not find {name}. Upload it as a Kaggle dataset.")
    matches = sorted(matches, key=lambda p: (len(str(p)), str(p)))
    return matches[0]

def find_bge_adapter():
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    candidates = []
    for root in roots:
        if root.exists():
            for p in root.rglob("adapter_config.json"):
                if (p.parent / "adapter_model.safetensors").exists():
                    # Prefer the saved BGE adapter path, not reranker adapters.
                    s = str(p.parent).lower()
                    if "bge" in s or "adapter" in s:
                        candidates.append(p.parent)
    if not candidates:
        raise FileNotFoundError(
            "Could not find BGE adapter folder containing adapter_config.json and adapter_model.safetensors. "
            "Upload Bgem3-finetune/bge-m3-health-qa/final as part of a Kaggle dataset."
        )
    candidates = sorted(candidates, key=lambda p: (("bge-m3-health-qa" not in str(p).lower()), len(str(p)), str(p)))
    return candidates[0]

TRAIN_PATH = find_file("Train.csv")
VAL_PATH = find_file("Val.csv")
TEST_PATH = find_file("Test.csv") if list(Path("/kaggle/input").rglob("Test.csv")) else None
BGE_ADAPTER = find_bge_adapter()

print("TRAIN_PATH", TRAIN_PATH)
print("VAL_PATH", VAL_PATH)
print("TEST_PATH", TEST_PATH)
print("BGE_ADAPTER", BGE_ADAPTER)

In [ ]:
qcol, acol, gcol, idcol = "input", "output", "subset", "ID"

train = pd.read_csv(TRAIN_PATH)
val = pd.read_csv(VAL_PATH)
for df in (train, val):
    for c in (qcol, acol, gcol):
        df[c] = df[c].fillna("").astype(str).str.strip()
train = train[(train[qcol] != "") & (train[acol] != "")].reset_index(drop=True)
val = val[(val[qcol] != "") & (val[acol] != "")].reset_index(drop=True)

print(f"train={len(train):,} val={len(val):,}")
print(train[gcol].value_counts().sort_index())

In [ ]:
class WhitespaceTokenizer:
    def tokenize(self, text):
        return [] if text is None else str(text).strip().split()

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    tokenizer=WhitespaceTokenizer(),
    use_stemmer=False,
)

def rouge_scores(pred, ref):
    s = scorer.score(str(ref), str(pred))
    return float(s["rouge1"].fmeasure), float(s["rougeL"].fmeasure)

def target_score(candidate_answer, reference):
    r1, rl = rouge_scores(candidate_answer, reference)
    return np.float32(0.75 * r1 + 0.25 * rl)

def tag_query(subset, query):
    return f"Subset: {subset}\nQuery: {query}"

def candidate_text(subset, q, a):
    return f"Subset: {subset}\nCandidate question: {q}\nCandidate answer: {a}"

In [ ]:
print("Loading fine-tuned BGE-M3 adapter for candidate generation...")
device = "cuda" if torch.cuda.is_available() else "cpu"
bi = SentenceTransformer("BAAI/bge-m3", device=device)
bi.max_seq_length = 256
inner = bi[0].auto_model
bi[0].auto_model = PeftModel.from_pretrained(inner, str(BGE_ADAPTER), is_trainable=False)
bi[0].auto_model.eval()

In [ ]:
def encode_subset_indices(df, name, k):
    indices = {}
    print(f"Encoding {name} per-subset query indices...")
    for subset, grp in tqdm(list(df.groupby(gcol)), desc=f"Index {name}"):
        embs = bi.encode(
            grp[qcol].tolist(),
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=128,
            convert_to_numpy=True,
        )
        nn = NearestNeighbors(n_neighbors=min(k + 1, len(grp)), metric="cosine").fit(embs)
        indices[subset] = {
            "nn": nn,
            "embs": embs,
            "q": np.array(grp[qcol].astype(str).tolist(), dtype=object),
            "a": np.array(grp[acol].astype(str).tolist(), dtype=object),
            "orig_idx": np.array(grp.index.tolist()),
        }
    return indices

def retrieve_topk(df, indices, k, leave_self_out=False):
    cands = [[] for _ in range(len(df))]
    pos = {idx: i for i, idx in enumerate(df.index)}
    for subset, grp in tqdm(list(df.groupby(gcol)), desc="Retrieve"):
        m = indices[subset]
        q_embs = bi.encode(
            grp[qcol].tolist(),
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=128,
            convert_to_numpy=True,
        )
        n_neighbors = min(k + (1 if leave_self_out else 0), len(m["a"]))
        _, idx_mat = m["nn"].kneighbors(q_embs, n_neighbors=n_neighbors)
        for row_idx, idxs in zip(grp.index, idx_mat):
            picked = []
            for j in idxs:
                if leave_self_out and m["orig_idx"][j] == row_idx:
                    continue
                picked.append({"q": str(m["q"][j]), "a": str(m["a"][j])})
                if len(picked) >= k:
                    break
            cands[pos[row_idx]] = picked
    return cands

k = CONFIG["k"]
train_idx = encode_subset_indices(train, "train", k)
train_cands = retrieve_topk(train, train_idx, k, leave_self_out=True)
val_cands = retrieve_topk(val, train_idx, k, leave_self_out=False)

In [ ]:
def evaluate_candidate_lists(cands, refs, subs, label):
    rows = []
    top1_preds = []
    oracle_preds = []
    for cs, ref, subset in zip(cands, refs, subs):
        if not cs:
            top1 = ""
            oracle = ""
            top1_r1 = oracle_r1 = 0.0
        else:
            top1 = cs[0]["a"]
            labels = [target_score(c["a"], ref) for c in cs]
            oracle = cs[int(np.argmax(labels))]["a"]
            top1_r1, _ = rouge_scores(top1, ref)
            oracle_r1, _ = rouge_scores(oracle, ref)
        top1_preds.append(top1)
        oracle_preds.append(oracle)
        rows.append({"subset": subset, "top1_r1": top1_r1, "oracle_r1": oracle_r1})
    df = pd.DataFrame(rows)
    per = df.groupby("subset")[["top1_r1", "oracle_r1"]].mean().round(4)
    out = {
        "label": label,
        "top1_r1": float(df["top1_r1"].mean()),
        "oracle_r1": float(df["oracle_r1"].mean()),
        "per_subset": per.to_dict(orient="index"),
    }
    print(json.dumps(out, indent=2))
    return out, top1_preds, oracle_preds

baseline_metrics, val_top1, val_oracle = evaluate_candidate_lists(
    val_cands, val[acol].tolist(), val[gcol].tolist(), f"ft_bgem3_per_subset_top{k}"
)

In [ ]:
print("Building subset-tagged cross-encoder regression pairs...")
rng = np.random.default_rng(SEED)
pair_q, pair_c, pair_y = [], [], []
per_subset_pair_counts = {}

for subset in sorted(train[gcol].unique()):
    subset_rows = np.where(train[gcol].to_numpy() == subset)[0]
    before = len(pair_y)
    for i in tqdm(subset_rows, desc=f"Pairs {subset}"):
        cs = train_cands[int(i)]
        if not cs:
            continue
        ref = train[acol].iloc[int(i)]
        labels = np.array([target_score(c["a"], ref) for c in cs], dtype=np.float32)
        order = np.argsort(-labels)
        chosen = []
        chosen.extend(order[: min(4, len(order))].tolist())
        chosen.extend(order[-min(4, len(order)) :].tolist())
        mid_pool = order[4:-4] if len(order) > 8 else order
        if len(mid_pool) > 0:
            n_mid = max(0, CONFIG["train_pairs_per_query"] - len(set(chosen)))
            chosen.extend(rng.choice(mid_pool, size=min(n_mid, len(mid_pool)), replace=False).tolist())
        seen = set()
        chosen = [x for x in chosen if not (x in seen or seen.add(x))]
        anchor = tag_query(subset, str(train[qcol].iloc[int(i)]))
        for j in chosen[: CONFIG["train_pairs_per_query"]]:
            pair_q.append(anchor)
            pair_c.append(candidate_text(subset, cs[int(j)]["q"], cs[int(j)]["a"]))
            pair_y.append(float(labels[int(j)]))
    per_subset_pair_counts[subset] = len(pair_y) - before

print("Pair counts:", per_subset_pair_counts)
print(f"Total pairs: {len(pair_y):,}")
print(pd.Series(pair_y).describe().round(4))

train_ds = Dataset.from_dict({"query": pair_q, "candidate": pair_c, "label": pair_y}).shuffle(seed=SEED)
del pair_q, pair_c, pair_y, train_cands
gc.collect()

In [ ]:
print("Freeing bi-encoder before cross-encoder training...")
try:
    bi.model.cpu()
except Exception:
    pass
del bi
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
print("Loading BGE reranker cross-encoder...")
reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3",
    num_labels=1,
    max_length=CONFIG["max_length"],
    device="cuda" if torch.cuda.is_available() else "cpu",
)
loss = MSELoss(model=reranker)

args = CrossEncoderTrainingArguments(
    output_dir=str(CONFIG["output_dir"] / "trainer"),
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    learning_rate=CONFIG["lr"],
    warmup_ratio=0.1,
    fp16=True,
    bf16=False,
    tf32=True,
    logging_steps=100,
    save_strategy="epoch",
    save_total_limit=1,
    report_to=["none"],
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    seed=SEED,
)

trainer = CrossEncoderTrainer(model=reranker, args=args, train_dataset=train_ds, loss=loss)
print("Starting subset-tagged cross-encoder training...")
t0 = time.time()
trainer.train()
train_seconds = time.time() - t0
print(f"Cross-encoder trained in {train_seconds / 3600:.2f} hours")
reranker.save_pretrained(str(CONFIG["output_dir"] / "final"))

In [ ]:
print("Scoring validation candidates with subset-tagged reranker...")
flat_pairs, row_lens = [], []
for q, subset, cs in zip(val[qcol].tolist(), val[gcol].tolist(), val_cands):
    tq = tag_query(subset, q)
    flat_pairs.extend([(tq, candidate_text(subset, c["q"], c["a"])) for c in cs])
    row_lens.append(len(cs))

scores = reranker.predict(flat_pairs, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
rerank_preds, chosen_ranks = [], []
off = 0
for cs, n in zip(val_cands, row_lens):
    if n == 0:
        rerank_preds.append("")
        chosen_ranks.append(0)
        continue
    row_scores = scores[off : off + n]
    off += n
    j = int(np.argmax(row_scores))
    rerank_preds.append(cs[j]["a"])
    chosen_ranks.append(j + 1)

In [ ]:
def score_preds(preds, refs, subs, label):
    rows = []
    for p, r, s_name in zip(preds, refs, subs):
        r1, rl = rouge_scores(p, r)
        rows.append({"subset": s_name, "rouge1": r1, "rougeL": rl})
    df = pd.DataFrame(rows)
    per = df.groupby("subset")[["rouge1", "rougeL"]].mean().round(4)
    out = {
        "label": label,
        "rouge1": float(df["rouge1"].mean()),
        "rougeL": float(df["rougeL"].mean()),
        "per_subset": per.to_dict(orient="index"),
    }
    print(json.dumps(out, indent=2))
    return out

refs = val[acol].tolist()
subs = val[gcol].tolist()
top1_metrics = score_preds(val_top1, refs, subs, "ft_bgem3_top1")
rerank_metrics = score_preds(rerank_preds, refs, subs, "subset_tag_crossencoder_rerank")
oracle_metrics = score_preds(val_oracle, refs, subs, "oracle_topk")

pred_df = pd.DataFrame({
    "ID": val[idcol].tolist(),
    "subset": val[gcol].tolist(),
    "top1": val_top1,
    "rerank": rerank_preds,
    "oracle": val_oracle,
    "reference": val[acol].tolist(),
    "chosen_rank": chosen_ranks,
})
pred_df.to_csv(CONFIG["output_dir"] / "val_predictions.csv", index=False)

summary = {
    "experiment": "subset_tagged_ft_bgem3_top50_crossencoder_rouge_regression",
    "platform": "kaggle",
    "intended_gpu": "T4/T4x2",
    "k": CONFIG["k"],
    "train_pairs_per_query": CONFIG["train_pairs_per_query"],
    "epochs": CONFIG["epochs"],
    "batch_size": CONFIG["batch_size"],
    "gradient_accumulation": CONFIG["grad_accum"],
    "effective_batch": CONFIG["batch_size"] * CONFIG["grad_accum"],
    "learning_rate": CONFIG["lr"],
    "train_seconds": train_seconds,
    "candidate_baseline": baseline_metrics,
    "top1": top1_metrics,
    "rerank": rerank_metrics,
    "oracle": oracle_metrics,
    "delta_rerank_vs_exp2": rerank_metrics["rouge1"] - 0.5892166283468145,
}
(CONFIG["output_dir"] / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("SUMMARY")
print(json.dumps(summary, indent=2))